# Policy Iteration implementation on Frozen Lake environment

# Create frozen lake environment

In [1]:
import random
import numpy as np

In [125]:
x = {}
x[(0, 1)] = 2
x

{(0, 1): 2}

In [544]:
class FrozenLakeEnvironment:
    def __init__(self, grid, slippery=True):
        self.grid = grid
        self.n_rows = len(grid)
        self.n_cols = len(grid[0])
        self.slippery = slippery
        self.possible_actions = [0, 1, 2, 3] # LEFT, DOWN, RIGHT, UP
        self.action_names = {
            "left": 0,
            "down": 1,
            "right": 2,
            "up": 3
        }
        
        self.action_mapper = {
            #[ ][ ][ ]
            #[ ][.][ ]
            #[ ][ ][ ]
            0: (0, -1), # left -> stay in the same row, move one column left
            1: (1, 0), # DOWN -> move on row down but stay in the same column
            2: (0, 1), # RIGHT -> same row but move one column right
            3: (-1, 0) # UP -> moe on row up but stay in the same column
        }
        
        self.start_state = self.__find("S")
        self.goal_state = self.__find("G")
        self.current_state = self.start_state

    def __find(self, value):
        for r in range(self.n_rows):
            for c in range(self.n_cols):
                if self.grid[r][c] == value:
                    return (r, c)

    def __is_terminal(self, state):
        r, c = state
        return self.grid[r][c] in ["G", "H"]

    def __move(self, state, action):
        r, c = state
        step_r, step_c = self.action_mapper[action]
        new_r = min(max(0, r+step_r), self.n_rows -1) #make sure step is a valid one
        new_c = min(max(0, c+step_c), self.n_cols -1)
        return new_r, new_c
        
    def get_transition_prob(self, state, action):
        """ returns list of (prob, next_step, reward, is_game_over) """
        if self.__is_terminal(state):
            return [{"prob": 1.0,
                     "new_state": state,
                     "reward": 0,
                     "game_over": True}]

        # --- deterministic action --------
        actions = [action]
        probs = [1.0]

        # --- stochastic action ----------
        if self.slippery: 
            actions = [action,
                       (action + 1)%len(self.possible_actions),
                       (action + 3)%len(self.possible_actions)]
            
            probs = [0.8,
                     0.1,
                     0.1]
        # --------------------------------
        transitions = []
        for action, prob in zip(actions, probs):
            new_state = self.__move(state, action)
            r, c = new_state
            cell = self.grid[r][c]
            
            if cell == "G":
                reward = 10
            elif cell == "H":
                reward = -10
            else:
                reward = -1
                
            game_over = cell in ["H", "G"]
            transitions.append({"prob": prob,
                                "new_state": new_state,
                                "reward": reward,
                                "game_over": game_over})
        return transitions

    def step(self, action):
        transitions = self.get_transition_prob(self.current_state, action)
        action_probs = [transition["prob"] for transition in transitions]
        selected_transition = random.choices(transitions, weights=action_probs)[0]
        return selected_transition
        
    def n_states(self):
        return self.n_rows * self.n_cols
    
    def n_actions(self):
        return len(self.actions)    

    def reset(self):
        self.current_state = self.start_state

In [566]:
lake_grid = [["S", "F", "F", "F"],
             ["F", "H", "F", "H"],
             ["F", "H", "F", "H"],
             ["H", "F", "F", "G"]]

In [567]:
frozen_lake = FrozenLakeEnvironment(grid=lake_grid,
                                        slippery=True)

In [568]:
frozen_lake.current_state

(0, 0)

In [569]:
current_state = (2, 2)
action_index = frozen_lake.action_names["right"]
frozen_lake._FrozenLakeEnvironment__move(current_state, action_index)

(2, 3)

In [570]:
frozen_lake.get_transition_prob(current_state, action_index)

[{'prob': 0.8, 'new_state': (2, 3), 'reward': -10, 'game_over': True},
 {'prob': 0.1, 'new_state': (1, 2), 'reward': -1, 'game_over': False},
 {'prob': 0.1, 'new_state': (3, 2), 'reward': -1, 'game_over': False}]

In [571]:
frozen_lake.step(action_index)

{'prob': 0.8, 'new_state': (0, 1), 'reward': -1, 'game_over': False}

# Policy Iteration Algorithm Implementation

In [572]:
n_rows = frozen_lake.n_rows
n_cols = frozen_lake.n_cols
policy = np.zeros((n_rows, n_cols), dtype=np.int8)
policy

array([[0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0]], dtype=int8)

In [573]:
V = np.zeros((n_rows, n_cols), dtype=np.float32)
V

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]], dtype=float32)

In [574]:
def policy_evaluation(environment, policy, V, gamma=0.99, theta=1e-8):
    while True:
        delta = 0
        for r in range(len(V)):
            for c in range(len(V[0])):
                if environment.grid[r][c] in ["H", "G"]:
                    continue 
                # state = (r, c)
                action = policy[r][c]
                new_v = 0
                for transition in environment.get_transition_prob((r,c), action):
                    nr, nc = transition["new_state"]
                    new_v += transition["prob"] * (transition["reward"] + gamma * V[nr][nc])
                    
                delta = max(delta, abs(V[r][c] - new_v))
                V[r][c] = new_v
                
        if delta < theta:
            break
    return V

In [575]:
policy_evaluation(frozen_lake, policy, V)

array([[-45.172836, -41.816742, -39.31176 , -36.664722],
       [-39.63478 ,   0.      , -13.141779,   0.      ],
       [-27.999245,   0.      , -10.605203,   0.      ],
       [  0.      , -10.099889, -11.153193,   0.      ]], dtype=float32)

In [576]:
def policy_improvement(environment, V, gamma=0.99):
    new_policy = np.zeros((environment.n_rows, environment.n_cols), dtype=np.int8)
    for r in range(len(V)):
        for c in range(len(V[0])):
            if environment.grid[r][c] in ["H", "G"]:
                    continue #
            # state = (r, c)
            action_values = []
            
            for action in environment.possible_actions:
                q = 0
                for transition in environment.get_transition_prob((r, c), action):
                    nr, nc = transition["new_state"]
                    q += transition["prob"] * (transition["reward"] + gamma * V[nr][nc])
                action_values.append(q)
            new_policy[r][c] = np.argmax(action_values).item()
    return new_policy    

In [577]:
def policy_iteration(environment):
    n_rows = environment.n_rows
    n_cols = environment.n_cols
    policy = np.zeros((n_rows, n_cols), dtype=np.int8)
    V = np.zeros((n_rows, n_cols), dtype=np.float32)
    while True:
        V = policy_evaluation(environment, policy, V)
        new_policy = policy_improvement(environment, V)
        if np.array_equal(policy, new_policy):
            break
            
        policy = new_policy
    return policy, V

In [578]:
policy, V = policy_iteration(frozen_lake)

In [579]:
policy

array([[2, 2, 1, 0],
       [3, 0, 1, 0],
       [3, 0, 1, 0],
       [0, 2, 2, 0]], dtype=int8)

In [580]:
text_policy = []
idx_to_name = {v:k for k, v in frozen_lake.action_names.items()}
print(idx_to_name)
for r in range(policy.shape[0]):
    col = []
    for c in range(policy.shape[1]):
        col.append(idx_to_name[policy[r][c].item()])
    text_policy.append(col)

{0: 'left', 1: 'down', 2: 'right', 3: 'up'}


In [581]:
text_policy

[['right', 'right', 'down', 'left'],
 ['up', 'left', 'down', 'left'],
 ['up', 'left', 'down', 'left'],
 ['left', 'right', 'right', 'left']]

# Render policy

In [582]:
import matplotlib.pyplot as plt

In [583]:
ARROWS = {
    0: "←",
    1: "↓",
    2: "→",
    3: "↑"
}

In [584]:
def render_policy(env, policy, agent_state=None):
    fig, ax = plt.subplots(figsize=(4, 4))
    
    # Grid properties
    rows, cols = env.n_rows, env.n_cols
    
    for r in range(rows):
        for c in range(cols):
            cell = env.grid[r][c]
            
            # 1. Determine aesthetic colors and labels
            facecolor = "white"
            label = ""
            alpha = 1.0
            
            if cell == "S":
                facecolor = "#a2d2ff"  # Soft Blue
                label = "START"
            elif cell == "F":
                facecolor = "#f8f9fa"  # Off-white
            elif cell == "H":
                facecolor = "#495057"  # Slate Gray (Hole)
                label = "🕳️"
            elif cell == "G":
                facecolor = "#b7e4c7"  # Soft Green (Goal)
                label = "🏁"

            # 2. Draw the cell
            rect = plt.Rectangle((c, rows - r - 1), 1, 1,
                                edgecolor="#ced4da", facecolor=facecolor, 
                                linewidth=1.5, alpha=alpha)
            ax.add_patch(rect)

            # 3. Draw arrows ONLY for walkable path (Start and Frozen tiles)
            if cell in ["F", "S"]:
                action = policy[r][c]
                ax.text(c + 0.5, rows - r - 0.5, ARROWS[action], 
                        ha="center", va="center", fontsize=20, 
                        color="#212529", fontweight='bold')
            
            # 4. Add text labels for H, G, or S
            if label:
                ax.text(c + 0.5, rows - r - 0.2, label, 
                        ha="center", va="center", fontsize=12, fontweight='bold')

In [585]:
import pandas as pd
from IPython.display import display

def render_policy_pretty(env, policy):
    rows, cols = env.n_rows, env.n_cols
    data = []
    
    # Mapping for styles and icons
    icons = {"S": "🚀", "H": "🕳️", "G": "🏁", "F": ""}
    arrows = {0: "←", 1: "↓", 2: "→", 3: "↑"}
    
    grid_display = []
    for r in range(rows):
        row_display = []
        for c in range(cols):
            tile = env.grid[r][c]
            if tile in ["S", "F"]:
                # Show arrow for navigable tiles
                action = policy[r][c]
                content = f"{icons[tile]} {arrows[action]}"
            else:
                # Show only icon for Hole/Goal
                content = icons[tile]
            row_display.append(content)
        grid_display.append(row_display)
    
    # Create DataFrame
    df = pd.DataFrame(grid_display)
    
    # Apply modern styling
    def style_cells(val):
        style = 'width: 60px; height: 60px; text-align: center; font-size: 20px; border: 1px solid #dee2e6;'
        if "🏁" in val: return style + 'background-color: #d4edda;' # Green
        if "🕳️" in val: return style + 'background-color: #f8d7da;' # Red/Gray
        if "🚀" in val: return style + 'background-color: #cce5ff;' # Blue
        return style + 'background-color: #89cfef;'

    styled_df = df.style.applymap(style_cells)
    display(styled_df)


In [586]:
render_policy_pretty(frozen_lake, policy)

/tmp/ipykernel_529021/616307430.py:38: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled_df = df.style.applymap(style_cells)


,0,1,2,3
0,🚀 →,→,↓,←
1,↑,🕳️,↓,🕳️
2,↑,🕳️,↓,🕳️
3,🕳️,→,→,🏁
